In [1]:
# 4_temperature_ablation.py
!pip install -q torch transformers sentence-transformers faiss-cpu pandas tqdm scikit-learn rouge-score nltk bert-score sacrebleu

# ------------------- IMPORTS -------------------
import pandas as pd, numpy as np, torch, faiss, time, nltk, warnings, logging
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score
from sacrebleu.metrics import BLEU

# Reduce HuggingFace model init warnings
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# NLTK downloads
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

warnings.filterwarnings("ignore")

# ------------------- DATA -------------------
df = pd.read_csv("/kaggle/input/mlops-amazon/amazon.csv")

documents = [
    f"""Product: {r['product_name']}
Price: {r['discounted_price']} | Rating: {r['rating']} ({r['rating_count']} reviews)
Description: {r['about_product']}"""
    for _, r in df.iterrows()
]

TEST_QUERIES = [
    {
        "query": "Recommend a good fast charging USB-C cable under 300 rupees",
        "reference": "The boAt A400 at ₹299 is recommended, supporting 24W-25W fast charging and having a slightly higher customer rating. For a tighter budget, the pTron Solero TB301 at ₹149 supports 15W fast charging.",
    },
    {
        "query": "Which cable has the highest rating and supports 60W charging?",
        "reference": "The Belkin USB-C to USB-C Fast Charging Cable (60W PD) is the best match, offering a strong 4.5-star rating along with full 60W fast-charging support.",
    },
    {
        "query": "What is the best iPhone lightning cable in the list?",
        "reference": "For super-fast charging with a USB-C adapter, the Belkin Lightning to USB-C is best. For a reliable and durable standard cable with a good warranty, the Duracell USB-A to Lightning is a great choice. The Hi-Mobiler is the most budget-friendly option.",
    },
    {
        "query": "Suggest me some good long lasting headphones",
        "reference": "The boAt Bassheads 100 in-ear wired earphones are recommended for longevity, featuring a premium coated wire for sturdiness and a 1-year warranty. They have a 4.1-star rating from over 360,000 reviews and cost between ₹349-₹379.",
    },
]


# ------------------- METRICS CLASS -------------------
class Metrics:
    def __init__(self):
        self.rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
        self.bleu = BLEU(effective_order=True)
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")

    def all(self, pred, ref, ctx):
        r = self.rouge.score(ref, pred)
        metrics = {
            "rouge_1_f1": r["rouge1"].fmeasure,
            "rouge_l_f1": r["rougeL"].fmeasure,
            "bleu": self.bleu.sentence_score(pred, [ref]).score / 100,
            "meteor": meteor_score([word_tokenize(ref.lower())], word_tokenize(pred.lower())),
        }
        P, R, F = bert_score(
            [pred], [ref], model_type="microsoft/deberta-large-mnli", verbose=False
        )
        metrics["bert_f1"] = F.mean().item()

        e1 = self.embedder.encode(pred)
        e2 = self.embedder.encode(ref)
        metrics["emb_sim"] = util.cos_sim(e1, e2).item()

        c_emb = self.embedder.encode(ctx)
        metrics["faith"] = min(1.0, 0.7 * util.cos_sim(e1, c_emb).item() + 0.3)

        return metrics

    def composite(self, m):
        w = {
            "rouge_1_f1": 0.1,
            "rouge_l_f1": 0.1,
            "bleu": 0.1,
            "meteor": 0.15,
            "bert_f1": 0.25,
            "emb_sim": 0.2,
            "faith": 0.1,
        }
        return sum(m[k] * w[k] for k in w)


metrics_calc = Metrics()


# ------------------- RAG CLASS -------------------
class RAG:
    def __init__(self, emb_name, generator):
        self.emb_name = emb_name
        self.generator = generator

        print(f"Loading embedding model: {emb_name}")
        self.embedder = SentenceTransformer(emb_name)

        dim = self.embedder.encode(["test"]).shape[1]
        self.index = faiss.IndexFlatIP(dim)

        print(f"Embedding {len(documents)} documents...")
        batches = [documents[i : i + 32] for i in range(0, len(documents), 32)]
        for b in tqdm(batches, desc="Indexing"):
            embs = self.embedder.encode(b, normalize_embeddings=True)
            self.index.add(embs)

    def retrieve(self, q, k):
        qe = self.embedder.encode([q], normalize_embeddings=True)
        D, I = self.index.search(qe, k)
        ctx = "\n\n".join([documents[i] for i in I[0]])
        return ctx


# ------------------- PATCH GENERATE METHOD -------------------
def generate(self, q, ctx, temperature=0.7):
    prompt = f"Context:\n{ctx}\n\nQuestion: {q}\nAnswer:"
    out = self.generator(
        prompt,
        max_new_tokens=512,
        temperature=temperature,
        top_p=0.95,
        top_k=50,
        do_sample=True,
    )[0]["generated_text"]
    ans = out.split("Answer:")[-1].strip()
    return ans


RAG.generate = generate

# ------------------- LOAD GENERATOR AND RAG -------------------
GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

print("\nLoading generator...")
generator = pipeline(
    "text-generation", model=GEN_MODEL, torch_dtype=torch.bfloat16, device_map="auto"
)

print("\nLoading RAG with fixed models...")
rag = RAG(EMBEDDING_MODEL, generator)

# ------------------- TEMPERATURE ABALATION EXPERIMENT -------------------
results = []
TEMPERATURES = [0.1, 0.3, 0.5, 0.7, 0.9, 1.2]

for temp in TEMPERATURES:
    print(f"\n{'='*80}\nTESTING TEMPERATURE: {temp}\n{'='*80}")
    for qd in TEST_QUERIES:
        ctx = rag.retrieve(qd["query"], k=5)
        ans = rag.generate(qd["query"], ctx, temperature=temp)
        m = metrics_calc.all(ans, qd["reference"], ctx)
        m["composite"] = metrics_calc.composite(m)
        results.append({**m, "temperature": temp, "query": qd["query"][:60]})
        print("\n------------------------------------------------------------")
        print(f"Temperature: {temp}")
        print(f"Query: {qd['query']}")
        print("\nGenerated Answer:")
        print(ans)
        print(f"\nComposite Score: {m['composite']:.4f}")
        print("------------------------------------------------------------\n")

df_out = pd.DataFrame(results)
summary = df_out.groupby("temperature")["composite"].mean().sort_values(ascending=False)

print("\n================ FINAL SUMMARY ================\n")
print("Average Composite Scores by Temperature:")
print(summary)

best_temp = summary.idxmax()
best_score = summary.max()
print(f"\n🏆 Best Temperature: {best_temp} → Composite Score: {best_score:.4f}")

df_out.to_csv("4_temperature_ablation.csv", index=False)
print("\nTemperature ablation results saved → 4_temperature_ablation.csv")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2025-12-05 07:38:09.531764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764920289.724845      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764920289.785509      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...


------------------------------------------------------------
Temperature: 1.2
Query: Which cable has the highest rating and supports 60W charging?

Generated Answer:
The MI Xiaomi USB Type C HYperCharge Cable 6A 100cm Sturdy and Durable Black is the product that has the highest rating at 4.3 out of 5 and supports 60W charging. If cost is a concern, there are two Ambrane options available as well, though they have slightly lower ratings. Here is a direct comparison:

- **MI Xiaomi Cable**: Rating - 4.3 (₹499), Supports - 120W (also 60W), High-quality design.
- **Ambrane Cable 1 (with 3A/60W, Black)**: Rating - 4.0 (₹179), Supports - 60W (up to 60W flash charge support).
- **Ambrane Cable 2 (with 3A/60W, Black)**: Rating - 4.0 (₹179), Supports - 60W (up to 60W flash charge support). The second Ambrane cable also meets your criteria, providing both high power output and a good rating at a more economical price.

Composite Score: 0.4227
----------------------------------------------------

The following layers were not sharded: encoder.layer.*.output.dense.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.attention.output.dense.weight, embeddings.LayerNorm.bias, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.self.pos_proj.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.dense.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.self.v_bias, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.self.in_proj.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.output.LayerNorm.weight, embeddings.LayerNorm.weight, encoder.layer.*.attention.self.pos_q_proj.bias, encoder.rel_embeddings.weight, encoder.layer.*.attention.self.pos_q_proj.weight, encoder.layer.*.attention.self.q_bias



------------------------------------------------------------
Temperature: 1.2
Query: What is the best iPhone lightning cable in the list?

Generated Answer:
Determining the "best" iPhone lightning cable largely depends on personal preferences and needs, but based on key factors like price, performance, compatibility, durability, and customer satisfaction:

1. **Hi-Mobiler**: Although it offers competitive pricing, its overall rating (4.0 out of 5 stars) is slightly lower compared to other options. It claims excellent materials, good durability, and a reasonable number of positive reviews.

2. **Belkin**: It stands out as the priciest option with a significantly higher rating of 4.4 out of 5 stars and boasts more positive reviews (1,951). It supports USB Power Delivery, offers fast charging, and is rated for a high number of bend and plug/unplug cycles, indicating superior durability. Its faster charging capabilities make it especially appealing for those who prioritize speed.

3. **Du

The following layers were not sharded: encoder.layer.*.output.dense.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.attention.output.dense.weight, embeddings.LayerNorm.bias, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.self.pos_proj.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.dense.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.self.v_bias, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.self.in_proj.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.output.LayerNorm.weight, embeddings.LayerNorm.weight, encoder.layer.*.attention.self.pos_q_proj.bias, encoder.rel_embeddings.weight, encoder.layer.*.attention.self.pos_q_proj.weight, encoder.layer.*.attention.self.q_bias



------------------------------------------------------------
Temperature: 1.2
Query: Suggest me some good long lasting headphones

Generated Answer:
For long-lasting headphones, I would recommend the boAt Rockerz 450 Bluetooth On Ear Headphones with Mic. These headphones offer an impressive 15 hours of playback time, ensuring you can enjoy music throughout the day and night without interruption. They come equipped with 40mm dynamic drivers that deliver immersive HD audio, providing a superior listening experience. Additionally, the ergonomic design and comfortable padded ear cushions make these headphones perfect for extended wear. The ability to switch between Bluetooth and AUX modes, along with integrated controls and dual connectivity options, adds to their versatility. Moreover, they feature a 1-year warranty for added peace of mind. The realme Buds Wireless might also be considered as an alternative due to their fast charging capabilities and up to 12 hours of playback time, alth